In [ ]:
# ===== Cell 1: Setup, font config, plotting utility =====
!apt-get -qq install -y fonts-liberation > /dev/null
!pip -q install matplotlib scikit-learn tensorflow

import os, shutil, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Register Liberation Serif (Times New Roman substitute) with matplotlib
liberation_path = "/usr/share/fonts/truetype/liberation/"
for f in os.listdir(liberation_path):
    if f.endswith(".ttf"):
        fm.fontManager.addfont(os.path.join(liberation_path, f))

plt.rcParams['font.family'] = 'Liberation Serif'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

FIG_DIR = "/content/exp5_figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_plot(fig, name, xlabel, ylabel, title=None):
    """Applies bold labels, Times New Roman/Liberation Serif, and saves as EPS + PDF at 600 DPI."""
    ax = fig.gca()
    ax.set_xlabel(xlabel, fontsize=14, fontweight='bold', fontname='Liberation Serif')
    ax.set_ylabel(ylabel, fontsize=14, fontweight='bold', fontname='Liberation Serif')
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold', fontname='Liberation Serif')
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname('Liberation Serif')
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"{name}.eps"), format='eps', dpi=600, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), format='pdf', dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {name}.eps and {name}.pdf")

print("Setup complete.")

In [ ]:
# ===== Cell 2: Load and prepare Oxford-IIIT Pet dataset (direct download, no tfds) =====
import tensorflow as tf
import numpy as np
import pathlib, re, tarfile
from sklearn.model_selection import KFold

IMG_SIZE = 224
BATCH = 32

DATA_DIR = pathlib.Path("/content/oxford_pets")
DATA_DIR.mkdir(exist_ok=True)

images_tar = tf.keras.utils.get_file(
    "images.tar.gz",
    origin="https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz",
    cache_dir=str(DATA_DIR), cache_subdir="."
)

img_dir = DATA_DIR / "images"
if not img_dir.exists():
    print("Extracting images...")
    with tarfile.open(images_tar) as tf_:
        tf_.extractall(DATA_DIR)

# Collect valid jpg files (skip the small number of corrupt/non-jpeg ones)
all_files = sorted([p for p in img_dir.glob("*.jpg")])

# Breed name = filename minus trailing _<number>.jpg, class = breed
breed_names = sorted(set(re.sub(r'_\d+\.jpg$', '', p.name) for p in all_files))
breed_to_idx = {b: i for i, b in enumerate(breed_names)}
NUM_CLASSES = len(breed_names)

filepaths = [str(p) for p in all_files]
labels = [breed_to_idx[re.sub(r'_\d+\.jpg$', '', p.name)] for p in all_files]

print(f"Found {len(filepaths)} images across {NUM_CLASSES} breeds.")

# Shuffle and split: 70% train, 15% val, 15% test
rng = np.random.RandomState(42)
idx = rng.permutation(len(filepaths))
filepaths = np.array(filepaths)[idx]
labels = np.array(labels)[idx]

n = len(filepaths)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_fp, train_lb = filepaths[:n_train], labels[:n_train]
val_fp, val_lb = filepaths[n_train:n_train+n_val], labels[n_train:n_train+n_val]
test_fp, test_lb = filepaths[n_train+n_val:], labels[n_train+n_val:]

def load_and_preprocess(path, label):
    raw = tf.io.read_file(path)
    image = tf.image.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def make_dataset(fps, lbs, batch_size=BATCH, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((fps, lbs))
    if shuffle:
        ds = ds.shuffle(len(fps), seed=42)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

ds_train = make_dataset(train_fp, train_lb, shuffle=True)
ds_val = make_dataset(val_fp, val_lb)
ds_test_p = make_dataset(test_fp, test_lb)

print(f"Classes: {NUM_CLASSES}, Train: {len(train_fp)}, Val: {len(val_fp)}, Test: {len(test_fp)}")

In [ ]:
# ===== Cell 3: MobileNetV2 model builder =====
def build_model(init='he_normal', dropout=0.5, use_bn=True, freeze_base=True,
                 lr=1e-3, optimizer_name='adam', l2_reg=0.0):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    base.trainable = not freeze_base

    reg = tf.keras.regularizers.l2(l2_reg) if l2_reg > 0 else None

    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, kernel_initializer=init, kernel_regularizer=reg)(x)
    if use_bn:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    if dropout > 0:
        x = tf.keras.layers.Dropout(dropout)(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer=init)(x)

    model = tf.keras.Model(base.input, out)

    if optimizer_name == 'sgd':
        opt = tf.keras.optimizers.SGD(learning_rate=lr)
    elif optimizer_name == 'momentum':
        opt = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    elif optimizer_name == 'rmsprop':
        opt = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print("Model builder ready.")

In [ ]:
# ===== Cell 4: Weight Initialization comparison =====
EPOCHS = 8
inits = {'Zero': 'zeros', 'Random': 'random_normal', 'Xavier': 'glorot_uniform', 'He': 'he_normal'}
init_histories = {}

for name, init_code in inits.items():
    print(f"Training with {name} initialization...")
    model = build_model(init=init_code, freeze_base=True)
    hist = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, verbose=1)
    init_histories[name] = hist.history

# Plot 1: Training Loss vs Epoch
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in init_histories.items():
    ax.plot(range(1, EPOCHS+1), h['loss'], marker='o', label=name)
ax.legend(title='Initialization')
save_plot(fig, "Plot1_TrainingLoss_Initialization", "Epoch", "Training Loss",
          "Training Loss vs. Epoch for Different Initializations")

# Plot 2: Validation Accuracy vs Epoch
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in init_histories.items():
    ax.plot(range(1, EPOCHS+1), [a*100 for a in h['val_accuracy']], marker='o', label=name)
ax.legend(title='Initialization')
save_plot(fig, "Plot2_ValAccuracy_Initialization", "Epoch", "Validation Accuracy (%)",
          "Validation Accuracy vs. Epoch for Different Initializations")

In [ ]:
# ===== Cell 5: Regularization comparison =====
reg_configs = {
    'No Regularization': dict(dropout=0.0, use_bn=False, l2_reg=0.0),
    'L2': dict(dropout=0.0, use_bn=False, l2_reg=1e-4),
    'Dropout': dict(dropout=0.5, use_bn=False, l2_reg=0.0),
    'BatchNorm': dict(dropout=0.0, use_bn=True, l2_reg=0.0),
}
reg_histories = {}

for name, cfg in reg_configs.items():
    print(f"Training with {name}...")
    model = build_model(init='he_normal', freeze_base=True, **cfg)
    hist = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, verbose=1)
    reg_histories[name] = hist.history

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# Plot 3: Train/Val Accuracy vs Epoch (using Dropout config as the example curve set)
fig, ax = plt.subplots(figsize=(7, 5))
h = reg_histories['Dropout']
ax.plot(range(1, EPOCHS+1), [a*100 for a in h['accuracy']], marker='o', label='Training')
ax.plot(range(1, EPOCHS+1), [a*100 for a in h['val_accuracy']], marker='s', label='Validation')
ax.legend()
save_plot(fig, "Plot3_TrainVal_Accuracy", "Epoch", "Accuracy (%)",
          "Training and Validation Accuracy vs. Epoch")

# Plot 4: Train/Val Loss vs Epoch
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(range(1, EPOCHS+1), h['loss'], marker='o', label='Training')
ax.plot(range(1, EPOCHS+1), h['val_loss'], marker='s', label='Validation')
ax.legend()
save_plot(fig, "Plot4_TrainVal_Loss", "Epoch", "Loss",
          "Training and Validation Loss vs. Epoch")

In [ ]:
# ===== Cell 6: With vs Without Batch Normalization =====
bn_yes = reg_histories['BatchNorm']
bn_no = reg_histories['No Regularization']

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(range(1, EPOCHS+1), [a*100 for a in bn_yes['val_accuracy']], marker='o', label='With BN')
ax.plot(range(1, EPOCHS+1), [a*100 for a in bn_no['val_accuracy']], marker='s', label='Without BN')
ax.legend()
save_plot(fig, "Plot5_BatchNorm_Comparison", "Epoch", "Validation Accuracy (%)",
          "With vs. Without Batch Normalization")

In [ ]:
# ===== Cell 7: Optimizer comparison =====
import time
optimizers = ['sgd', 'momentum', 'rmsprop', 'adam']
opt_histories = {}
opt_table_rows = []

for opt_name in optimizers:
    print(f"Training with {opt_name}...")
    model = build_model(init='he_normal', freeze_base=True, optimizer_name=opt_name)
    t0 = time.time()
    hist = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, verbose=1)
    elapsed = time.time() - t0
    opt_histories[opt_name] = hist.history
    best_val = max(hist.history['val_accuracy']) * 100
    epoch_conv = int(np.argmax(hist.history['val_accuracy'])) + 1
    opt_table_rows.append([opt_name.upper(), hist.history['loss'][-1], best_val, epoch_conv, elapsed])

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# Plot 6: Training Loss vs Epoch
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in opt_histories.items():
    ax.plot(range(1, EPOCHS+1), h['loss'], marker='o', label=name.upper())
ax.legend(title='Optimizer')
save_plot(fig, "Plot6_TrainingLoss_Optimizers", "Epoch", "Training Loss",
          "Training Loss vs. Epoch for Different Optimizers")

# Plot 7: Validation Accuracy vs Epoch
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in opt_histories.items():
    ax.plot(range(1, EPOCHS+1), [a*100 for a in h['val_accuracy']], marker='o', label=name.upper())
ax.legend(title='Optimizer')
save_plot(fig, "Plot7_ValAccuracy_Optimizers", "Epoch", "Validation Accuracy (%)",
          "Validation Accuracy vs. Epoch for Different Optimizers")

# Print the results table (paste this into your LaTeX table)
print(f"{'Optimizer':<10}{'Final Loss':<12}{'Best Val Acc':<14}{'Epoch Conv.':<12}{'Time (s)'}")
for row in opt_table_rows:
    print(f"{row[0]:<10}{row[1]:<12.4f}{row[2]:<14.2f}{row[3]:<12}{row[4]:.2f}")

In [ ]:
# ===== Cell 8: Hyperparameter tuning (LR, Batch Size, Dropout) =====
EPOCHS_HP = 5

# Learning rate sweep
lr_results = {}
for lr in [0.001, 0.0001]:
    model = build_model(init='he_normal', lr=lr, freeze_base=True)
    hist = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_HP, verbose=1)
    lr_results[lr] = max(hist.history['val_accuracy']) * 100

    del model
    tf.keras.backend.clear_session()
    gc.collect()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(list(lr_results.keys()), list(lr_results.values()), marker='o', linewidth=2)
ax.set_xscale('log')
save_plot(fig, "Plot8_LearningRate_ValAccuracy", "Learning Rate", "Validation Accuracy (%)",
          "Learning Rate vs. Validation Accuracy")

# Batch size sweep
bs_results = {}
for bs in [16, 32, 64]:
    tr = make_dataset(train_fp, train_lb, batch_size=bs, shuffle=True)
    va = make_dataset(val_fp, val_lb, batch_size=bs)
    model = build_model(init='he_normal', freeze_base=True)
    hist = model.fit(tr, validation_data=va, epochs=EPOCHS_HP, verbose=1)
    bs_results[bs] = max(hist.history['val_accuracy']) * 100

    del model, tr, va
    tf.keras.backend.clear_session()
    gc.collect()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(list(bs_results.keys()), list(bs_results.values()), marker='o', linewidth=2)
save_plot(fig, "Plot9_BatchSize_ValAccuracy", "Batch Size", "Validation Accuracy (%)",
          "Batch Size vs. Validation Accuracy")

# Dropout sweep
do_results = {}
for do in [0.0, 0.25, 0.5]:
    model = build_model(init='he_normal', dropout=do, freeze_base=True)
    hist = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_HP, verbose=1)
    do_results[do] = max(hist.history['val_accuracy']) * 100

    del model
    tf.keras.backend.clear_session()
    gc.collect()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(list(do_results.keys()), list(do_results.values()), marker='o', linewidth=2)
save_plot(fig, "Plot10_Dropout_ValAccuracy", "Dropout Rate", "Validation Accuracy (%)",
          "Dropout Rate vs. Validation Accuracy")

In [ ]:
# ===== Cell 9: Feature Extraction vs Fine-Tuning =====
EPOCHS_TL = 8

# Case A: Feature extraction (frozen base)
model_fe = build_model(init='he_normal', freeze_base=True, lr=1e-3)
hist_fe = model_fe.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_TL, verbose=1)
del model_fe
tf.keras.backend.clear_session()
gc.collect()

# Case B: Fine-tuning (unfreeze top layers, small LR)
model_ft = build_model(init='he_normal', freeze_base=True, lr=1e-3)
model_ft.fit(ds_train, validation_data=ds_val, epochs=3, verbose=1)  # warm up classifier first

base_layer = model_ft.layers[0] if hasattr(model_ft.layers[0], 'trainable') else None
for layer in model_ft.layers:
    if 'block_16' in layer.name or 'block_15' in layer.name or 'Conv_1' in layer.name:
        layer.trainable = True

model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_ft = model_ft.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_TL, verbose=1)

# Plot 11: Feature Extraction vs Fine-Tuning
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(range(1, EPOCHS_TL+1), [a*100 for a in hist_fe.history['val_accuracy']], marker='o', label='Feature Extraction')
ax.plot(range(1, EPOCHS_TL+1), [a*100 for a in hist_ft.history['val_accuracy']], marker='s', label='Fine-Tuning')
ax.legend()
save_plot(fig, "Plot11_FeatureExtraction_vs_FineTuning", "Epoch", "Validation Accuracy (%)",
          "Feature Extraction vs. Fine-Tuning")

# Plot 12: Loss comparison before/after fine-tuning
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(range(1, EPOCHS_TL+1), hist_fe.history['val_loss'], marker='o', label='Feature Extraction')
ax.plot(range(1, EPOCHS_TL+1), hist_ft.history['val_loss'], marker='s', label='Fine-Tuning')
ax.legend()
save_plot(fig, "Plot12_TrainVal_Loss_FineTuning", "Epoch", "Validation Loss",
          "Validation Loss Before and After Fine-Tuning")

# NOTE: model_ft is kept alive intentionally — Cell 11 doesn't use it,
# but if you re-run Cell 9 alone later, clear it manually first:
# del model_ft; tf.keras.backend.clear_session(); gc.collect()

In [ ]:
# ===== Cell 10: 5-Fold Cross-Validation on candidate configurations =====
X_all = train_fp   # array of file paths
y_all = train_lb   # array of integer labels

configs = {
    'C1': dict(dropout=0.5, use_bn=True, optimizer_name='adam', lr=1e-3),
    'C2': dict(dropout=0.25, use_bn=True, optimizer_name='adam', lr=1e-4),
    'C3': dict(dropout=0.5, use_bn=False, optimizer_name='rmsprop', lr=1e-3),
    'C4': dict(dropout=0.0, use_bn=True, optimizer_name='sgd', lr=1e-3),
}

EPOCHS_CV = 4
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for cname, cfg in configs.items():
    fold_accs = []
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):
        print(f"{cname} - Fold {fold_idx+1}/5")
        Xtr, Xva = X_all[train_idx], X_all[val_idx]
        ytr, yva = y_all[train_idx], y_all[val_idx]
        tr_ds = make_dataset(Xtr, ytr, shuffle=True)
        va_ds = make_dataset(Xva, yva)
        model = build_model(init='he_normal', freeze_base=True, **cfg)
        model.fit(tr_ds, validation_data=va_ds, epochs=EPOCHS_CV, verbose=0)
        _, acc = model.evaluate(va_ds, verbose=0)
        fold_accs.append(acc * 100)

        del model, tr_ds, va_ds
        tf.keras.backend.clear_session()
        gc.collect()

    cv_results[cname] = fold_accs
    print(f"{cname}: {np.mean(fold_accs):.2f} ± {np.std(fold_accs):.2f}")

# Plot 13: 5-Fold CV Accuracy with error bars
fig, ax = plt.subplots(figsize=(7, 5))
names = list(cv_results.keys())
means = [np.mean(cv_results[n]) for n in names]
stds = [np.std(cv_results[n]) for n in names]
ax.bar(names, means, yerr=stds, capsize=6)
save_plot(fig, "Plot13_KFold_CV_Accuracy", "Hyperparameter Configuration", "Mean Validation Accuracy (%)",
          "5-Fold Cross-Validation Accuracy")

best_config_name = names[int(np.argmax(means))]
print(f"Best configuration: {best_config_name}")

In [ ]:
# ===== Cell 11: Final Model Evaluation and Confusion Matrix =====
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import seaborn as sns

best_cfg = configs[best_config_name]
final_model = build_model(init='he_normal', freeze_base=True, **best_cfg)

t0 = time.time()
final_hist = final_model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_TL, verbose=1)
train_time = time.time() - t0

test_loss, test_acc = final_model.evaluate(ds_test_p, verbose=1)

y_true, y_pred = [], []
for imgs, lbls in ds_test_p:
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(lbls.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
n_params = final_model.count_params()

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
print(f"Training Time: {train_time:.2f}s, Parameters: {n_params:,}")

# Plot 14: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(cm, cmap='Blues', ax=ax, cbar=True, square=True)
save_plot(fig, "Plot14_ConfusionMatrix", "Predicted Class", "True Class",
          "Confusion Matrix on Test Set")

In [ ]:
# ===== Cell 12: Zip figures and download =====
from google.colab import files

zip_path = "/content/Exp5_Figures.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(FIG_DIR)):
        zf.write(os.path.join(FIG_DIR, fname), fname)

print(f"Zipped {len(os.listdir(FIG_DIR))} files into {zip_path}")
files.download(zip_path)